In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

# Current profiling run
run_id = (
    spark.table("workspace.synthetic_profile.current_run")
    .select("run_id")
    .first()["run_id"]
)

# Numeric distributions
numeric_stats = (
    spark.table("workspace.synthetic_profile.v_numeric_statistics")
    .toPandas()
)

# Categorical distributions
category_stats = (
    spark.table("workspace.synthetic_profile.v_category_statistics")
    .toPandas()
)

# Date statistics
date_stats = (
    spark.table("workspace.synthetic_profile.date_profile")
    .filter(F.col("run_id") == run_id)
    .toPandas()
)

# Null rates / data quality
column_quality = (
    spark.table("workspace.synthetic_profile.column_quality")
    .filter(F.col("run_id") == run_id)
    .toPandas()
)

# Relationships / correlations
dependency_stats = (
    spark.table("workspace.synthetic_profile.v_dependency_statistics")
    .toPandas()
)

# Structural relationships
structural_stats = (
    spark.table("workspace.synthetic_profile.v_structural_statistics")
    .toPandas()
)

# Conditional distributions
conditional_stats = (
    spark.table("workspace.synthetic_profile.conditional_profile")
    .filter(F.col("run_id") == run_id)
    .toPandas()
)

print("Profiles loaded.")

print("Numeric:", len(numeric_stats))
print("Categorical:", len(category_stats))
print("Dates:", len(date_stats))
print("Quality:", len(column_quality))
print("Dependencies:", len(dependency_stats))
print("Structural:", len(structural_stats))
print("Conditional:", len(conditional_stats))

Set n count for creation

In [0]:
N_CUSTOMERS = 10_000

SEED = 42

rng = np.random.default_rng(SEED)

print(f"Generating {N_CUSTOMERS:,} customers")

create synthetic customers

In [0]:
person_ids = [
    f"P{i:08d}"
    for i in range(1, N_CUSTOMERS + 1)
]

person_df = pd.DataFrame({
    "PARTY_ID": person_ids
})

person_df.head()

In [0]:
def generate_categorical(
    table_name,
    column_name,
    n
):
    profile = category_stats[
        (category_stats["logical_table_name"] == table_name)
        &
        (category_stats["column_name"] == column_name)
    ].copy()

    # handle NULLs separately later
    profile = profile[
        profile["category_value"].notna()
    ]

    if profile.empty:
        return np.array([None] * n, dtype=object)

    values = profile["category_value"].to_numpy()

    probabilities = (
        profile["proportion"]
        .astype(float)
        .to_numpy()
    )

    # Re-normalize after removing NULL category
    probabilities = (
        probabilities /
        probabilities.sum()
    )

    return rng.choice(
        values,
        size=n,
        p=probabilities
    )

In [0]:
person_categorical_columns = (
    category_stats[
        category_stats["logical_table_name"] == "PERSON"
    ]["column_name"]
    .unique()
)

for column_name in person_categorical_columns:

    if column_name == "PARTY_ID":
        continue

    person_df[column_name] = generate_categorical(
        "PERSON",
        column_name,
        N_CUSTOMERS
    )

person_df.head()

Generate numeric tables

In [0]:
def generate_numeric(
    table_name,
    column_name,
    n
):
    profile = numeric_stats[
        (numeric_stats["logical_table_name"] == table_name)
        &
        (numeric_stats["column_name"] == column_name)
    ]

    if profile.empty:
        return np.full(n, np.nan)

    row = profile.iloc[0]

    percentile_points = np.array([
        0.00,
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
        1.00
    ])

    values = np.array([
        row["min_value"],
        row["p01"],
        row["p05"],
        row["p25"],
        row["p50"],
        row["p75"],
        row["p95"],
        row["p99"],
        row["max_value"]
    ], dtype=float)

    # Remove missing percentile points
    valid = ~np.isnan(values)

    percentile_points = percentile_points[valid]
    values = values[valid]

    if len(values) == 0:
        return np.full(n, np.nan)

    if len(values) == 1:
        return np.full(n, values[0])

    random_percentiles = rng.random(n)

    return np.interp(
        random_percentiles,
        percentile_points,
        values
    )

In [0]:
person_numeric_columns = (
    numeric_stats[
        numeric_stats["logical_table_name"] == "PERSON"
    ]["column_name"]
    .unique()
)

for column_name in person_numeric_columns:

    if column_name in person_df.columns:
        continue

    person_df[column_name] = generate_numeric(
        "PERSON",
        column_name,
        N_CUSTOMERS
    )

Null probablity

In [0]:
def get_null_probability(
    table_name,
    column_name
):
    profile = column_quality[
        (column_quality["logical_table_name"] == table_name)
        &
        (column_quality["column_name"] == column_name)
    ]

    if profile.empty:
        return 0.0

    null_pct = profile.iloc[0]["null_pct"]

    if pd.isna(null_pct):
        return 0.0

    return float(null_pct) / 100

In [0]:
def apply_profiled_nulls(
    dataframe,
    table_name,
    protected_columns=None
):
    if protected_columns is None:
        protected_columns = []

    for column_name in dataframe.columns:

        if column_name in protected_columns:
            continue

        null_probability = get_null_probability(
            table_name,
            column_name
        )

        if null_probability <= 0:
            continue

        mask = (
            rng.random(len(dataframe))
            < null_probability
        )

        dataframe.loc[
            mask,
            column_name
        ] = None

    return dataframe

In [0]:
person_df = apply_profiled_nulls(
    person_df,
    "PERSON",
    protected_columns=["PARTY_ID"]
)

In [0]:
def generate_date(
    table_name,
    column_name,
    n
):
    profile = date_stats[
        (date_stats["logical_table_name"] == table_name)
        &
        (date_stats["column_name"] == column_name)
    ]

    if profile.empty:
        return pd.Series(
            [pd.NaT] * n
        )

    row = profile.iloc[0]

    earliest = pd.to_datetime(
        row["earliest_date"]
    )

    latest = pd.to_datetime(
        row["latest_date"]
    )

    if pd.isna(earliest) or pd.isna(latest):
        return pd.Series(
            [pd.NaT] * n
        )

    total_days = (
        latest - earliest
    ).days

    if total_days <= 0:
        return pd.Series(
            [earliest] * n
        )

    random_days = rng.integers(
        0,
        total_days + 1,
        size=n
    )

    return (
        earliest
        + pd.to_timedelta(
            random_days,
            unit="D"
        )
    )

In [0]:
person_date_columns = (
    date_stats[
        date_stats["logical_table_name"] == "PERSON"
    ]["column_name"]
    .unique()
)

for column_name in person_date_columns:

    person_df[column_name] = generate_date(
        "PERSON",
        column_name,
        N_CUSTOMERS
    )

In [0]:
if (
    "BIRTH_DT" in person_df.columns
    and
    "SINCE_DT" in person_df.columns
):

    invalid = (
        person_df["SINCE_DT"].notna()
        &
        person_df["BIRTH_DT"].notna()
        &
        (
            person_df["SINCE_DT"]
            <
            person_df["BIRTH_DT"]
        )
    )

    person_df.loc[
        invalid,
        "SINCE_DT"
    ] = person_df.loc[
        invalid,
        "BIRTH_DT"
    ]

Cardinality

In [0]:
def generate_cardinality(
    relationship_name,
    n,
    fallback=1
):
    profile = structural_stats[
        structural_stats["relationship_name"]
        == relationship_name
    ].copy()

    if profile.empty:
        return np.full(
            n,
            fallback,
            dtype=int
        )

    values = (
        profile["cardinality"]
        .astype(int)
        .to_numpy()
    )

    probabilities = (
        profile["proportion"]
        .astype(float)
        .to_numpy()
    )

    probabilities = (
        probabilities /
        probabilities.sum()
    )

    return rng.choice(
        values,
        size=n,
        p=probabilities
    )

Accounts per each customer

In [0]:
account_counts = generate_cardinality(
    "ACCOUNTS_PER_PARTY",
    N_CUSTOMERS,
    fallback=1
)

total_accounts = int(
    account_counts.sum()
)

print(
    f"{N_CUSTOMERS:,} customers"
)

print(
    f"{total_accounts:,} accounts"
)

In [0]:
account_ids = [
    f"A{i:09d}"
    for i in range(
        1,
        total_accounts + 1
    )
]

In [0]:
account_party_ids = np.repeat(
    person_df["PARTY_ID"].to_numpy(),
    account_counts
)

customer_account_rel_df = pd.DataFrame({
    "PARTY_ID": account_party_ids,
    "ACCT_ID": account_ids
})

In [0]:
account_df = pd.DataFrame({
    "ACCOUNT_ID": account_ids
})

Categorical col's

In [0]:
account_categorical_columns = (
    category_stats[
        category_stats["logical_table_name"]
        == "TKQ_ACCOUNT"
    ]["column_name"]
    .unique()
)

for column_name in account_categorical_columns:

    if column_name == "ACCOUNT_ID":
        continue

    account_df[column_name] = (
        generate_categorical(
            "TKQ_ACCOUNT",
            column_name,
            total_accounts
        )
    )

Numeric col's

In [0]:
account_numeric_columns = (
    numeric_stats[
        numeric_stats["logical_table_name"]
        == "TKQ_ACCOUNT"
    ]["column_name"]
    .unique()
)

for column_name in account_numeric_columns:

    if column_name in account_df.columns:
        continue

    account_df[column_name] = (
        generate_numeric(
            "TKQ_ACCOUNT",
            column_name,
            total_accounts
        )
    )

In [0]:
account_balance_df = pd.DataFrame({
    "ACCOUNT_ID": account_ids
})

In [0]:
raw_balance = generate_numeric(
    "TKQ_ACCOUNT_BALANCE",
    "CURRENT_BALANCE_AMT",
    total_accounts
)

raw_balance = np.maximum(
    raw_balance,
    0
)

In [0]:
if "CREDIT_LIMIT_AMT" in account_df.columns:

    credit_limit = (
        pd.to_numeric(
            account_df["CREDIT_LIMIT_AMT"],
            errors="coerce"
        )
        .fillna(0)
        .to_numpy()
    )

    # Allow some over-limit accounts,
    # but prevent absurd combinations
    current_balance = np.minimum(
        raw_balance,
        credit_limit * 1.25
    )

else:

    credit_limit = np.zeros(
        total_accounts
    )

    current_balance = raw_balance

In [0]:
account_balance_df[
    "CURRENT_BALANCE_AMT"
] = current_balance

In [0]:
account_balance_df[
    "AVAILABLE_TO_BUY_AMT"
] = np.maximum(
    credit_limit - current_balance,
    0
)

account_balance_df[
    "OVERLIMIT_AMT"
] = np.maximum(
    current_balance - credit_limit,
    0
)

In [0]:
account_df[
    "_UTILIZATION_RATE"
] = np.where(
    credit_limit > 0,
    current_balance / credit_limit,
    0
)

In [0]:
utilization_profile = conditional_stats[
    (
        conditional_stats["entity_name"]
        == "ACCOUNT"
    )
    &
    (
        conditional_stats["condition_variable"]
        == "UTILIZATION_RATE"
    )
    &
    (
        conditional_stats["outcome_variable"]
        == "PAST_DUE_INDICATO_CD"
    )
].copy()

In [0]:
delinquency_rates = {}

for _, row in utilization_profile.iterrows():

    delinquency_rates[
        row["condition_bucket"]
    ] = float(
        row["outcome_rate"]
    )

delinquency_rates

In [0]:
def get_delinquency_probability(
    utilization
):

    if utilization < 0.30:
        bucket = "0-30%"

    elif utilization < 0.60:
        bucket = "30-60%"

    elif utilization < 0.80:
        bucket = "60-80%"

    elif utilization <= 1.00:
        bucket = "80-100%"

    else:
        bucket = "100%+"

    # Use SQL rate if available
    if bucket in delinquency_rates:
        return delinquency_rates[bucket]

    # Temporary fallback
    return 0.05

In [0]:
delinquency_probability = (
    account_df["_UTILIZATION_RATE"]
    .apply(
        get_delinquency_probability
    )
    .to_numpy()
)

In [0]:
is_delinquent = (
    rng.random(total_accounts)
    <
    delinquency_probability
)

account_df[
    "PAST_DUE_INDICATO_CD"
] = np.where(
    is_delinquent,
    "Y",
    "N"
)

In [0]:
generated_dlq_count = generate_numeric(
    "TKQ_ACCOUNT",
    "DELINQUENCY_CNT",
    total_accounts
)

generated_dlq_count = np.nan_to_num(
    generated_dlq_count,
    nan=0
)

generated_dlq_count = np.maximum(
    np.round(
        generated_dlq_count
    ),
    1
)

account_df[
    "DELINQUENCY_CNT"
] = np.where(
    is_delinquent,
    generated_dlq_count,
    0
).astype(int)

In [0]:
def fill_profiled_fields(
    dataframe,
    table_name,
    protected_columns=None
):

    if protected_columns is None:
        protected_columns = []

    n = len(dataframe)

    # -----------------------
    # Categorical fields
    # -----------------------

    cat_columns = (
        category_stats[
            category_stats["logical_table_name"]
            == table_name
        ]["column_name"]
        .unique()
    )

    for column_name in cat_columns:

        if (
            column_name in protected_columns
            or
            column_name in dataframe.columns
        ):
            continue

        dataframe[column_name] = (
            generate_categorical(
                table_name,
                column_name,
                n
            )
        )

    # -----------------------
    # Numeric fields
    # -----------------------

    num_columns = (
        numeric_stats[
            numeric_stats["logical_table_name"]
            == table_name
        ]["column_name"]
        .unique()
    )

    for column_name in num_columns:

        if (
            column_name in protected_columns
            or
            column_name in dataframe.columns
        ):
            continue

        dataframe[column_name] = (
            generate_numeric(
                table_name,
                column_name,
                n
            )
        )

    # -----------------------
    # Date fields
    # -----------------------

    dt_columns = (
        date_stats[
            date_stats["logical_table_name"]
            == table_name
        ]["column_name"]
        .unique()
    )

    for column_name in dt_columns:

        if (
            column_name in protected_columns
            or
            column_name in dataframe.columns
        ):
            continue

        dataframe[column_name] = (
            generate_date(
                table_name,
                column_name,
                n
            )
        )

    dataframe = apply_profiled_nulls(
        dataframe,
        table_name,
        protected_columns
    )

    return dataframe

In [0]:
customer_account_rel_df = (
    fill_profiled_fields(
        customer_account_rel_df,
        "CUSTOMER_ACCOUNT_REL",
        protected_columns=[
            "PARTY_ID",
            "ACCT_ID"
        ]
    )
)

In [0]:
account_balance_df = (
    fill_profiled_fields(
        account_balance_df,
        "TKQ_ACCOUNT_BALANCE",
        protected_columns=[
            "ACCOUNT_ID",
            "CURRENT_BALANCE_AMT",
            "AVAILABLE_TO_BUY_AMT",
            "OVERLIMIT_AMT"
        ]
    )
)

In [0]:
tradeline_counts = generate_cardinality(
    "TRADELINES_PER_CIF",
    N_CUSTOMERS,
    fallback=3
)

total_tradelines = int(
    tradeline_counts.sum()
)

In [0]:
tradeline_party_ids = np.repeat(
    person_df["PARTY_ID"].to_numpy(),
    tradeline_counts
)

In [0]:
tradeline_ids = [
    f"TL{i:010d}"
    for i in range(
        1,
        total_tradelines + 1
    )
]

In [0]:
tradelines_df = pd.DataFrame({
    "REQ-CIF-KEY":
        tradeline_party_ids,

    "REQ-CIF-TIE-BREAKER":
        1,

    "account_number":
        tradeline_ids
})

In [0]:
tradelines_df = fill_profiled_fields(
    tradelines_df,
    "TRADELINES",
    protected_columns=[
        "REQ-CIF-KEY",
        "REQ-CIF-TIE-BREAKER",
        "account_number"
    ]
)

In [0]:
customer_relationship_df = pd.DataFrame({
    "FROM_PARTY_ID": pd.Series(
        dtype="object"
    ),
    "TO_PARTY_ID": pd.Series(
        dtype="object"
    )
})

In [0]:
tables = {
    "PERSON":
        person_df,

    "CUSTOMER_ACCOUNT_REL":
        customer_account_rel_df,

    "CUSTOMER_RELATIONSHIP":
        customer_relationship_df,

    "TKQ_ACCOUNT":
        account_df,

    "TKQ_ACCOUNT_BALANCE":
        account_balance_df,

    "TRADELINES":
        tradelines_df
}

In [0]:
for table_name, df in tables.items():

    print(
        f"{table_name}: "
        f"{len(df):,} rows"
    )

In [0]:
from pyspark.sql import functions as F

def add_missing_columns(df, logical_table_name):

    expected = (
        spark.table(
            "workspace.synthetic_profile.column_inventory"
        )
        .filter(
            F.col("logical_table_name")
            == logical_table_name
        )
        .select(
            "column_name",
            "data_type"
        )
        .collect()
    )

    for row in expected:

        column_name = row["column_name"]

        if column_name not in df.columns:
            df[column_name] = None

    return df

In [0]:
person_df = add_missing_columns(
    person_df,
    "PERSON"
)

account_df = add_missing_columns(
    account_df,
    "TKQ_ACCOUNT"
)

account_balance_df = add_missing_columns(
    account_balance_df,
    "TKQ_ACCOUNT_BALANCE"
)

customer_account_rel_df = add_missing_columns(
    customer_account_rel_df,
    "CUSTOMER_ACCOUNT_REL"
)

tradelines_df = add_missing_columns(
    tradelines_df,
    "TRADELINES"
)

In [0]:
def save_generated_table(df, table_name):
    (
        spark.createDataFrame(df)
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"workspace.default.{table_name}"
        )
    )


save_generated_table(
    person_df,
    "PERSON"
)

save_generated_table(
    customer_account_rel_df,
    "CUSTOMER_ACCOUNT_REL"
)

save_generated_table(
    account_df,
    "TKQ_ACCOUNT"
)

save_generated_table(
    account_balance_df,
    "TKQ_ACCOUNT_BALANCE"
)

save_generated_table(
    tradelines_df,
    "TRADELINES"
)

In [0]:
import os
import zipfile
import shutil

# Existing CSVs in the Volume
OUTPUT_DIR = "/Volumes/workspace/default/scotia/generated_row_level"

csv_paths = [
    f"{OUTPUT_DIR}/PERSON.csv",
    f"{OUTPUT_DIR}/CUSTOMER_ACCOUNT_REL.csv",
    f"{OUTPUT_DIR}/CUSTOMER_RELATIONSHIP.csv",
    f"{OUTPUT_DIR}/TKQ_ACCOUNT.csv",
    f"{OUTPUT_DIR}/TKQ_ACCOUNT_BALANCE.csv",
    f"{OUTPUT_DIR}/TRADELINES.csv"
]

# ---------------------------------------------------------
# Create ZIP locally first
# ---------------------------------------------------------

local_zip_path = "/tmp/RCAR_generated_row_level.zip"

with zipfile.ZipFile(
    local_zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for csv_path in csv_paths:

        zip_file.write(
            csv_path,
            arcname=os.path.basename(csv_path)
        )

print("Local ZIP created:")
print(local_zip_path)

# ---------------------------------------------------------
# Copy completed ZIP into the Databricks Volume
# ---------------------------------------------------------

volume_zip_path = (
    "/Volumes/workspace/default/scotia/"
    "RCAR_generated_row_level.zip"
)

shutil.copyfile(
    local_zip_path,
    volume_zip_path
)

print("\nZIP copied to Volume:")
print(volume_zip_path)